## Imports

In [6]:
import sys
project_root = "/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(sys.path[:3])

['/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project', '/home/krosen5/anaconda3/envs/legalsequencetransformer/lib/python313.zip', '/home/krosen5/anaconda3/envs/legalsequencetransformer/lib/python3.13']


In [7]:
import os
print(os.getcwd())
print(os.listdir("."))

/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project/side_tests
['load_in_model_just_clustering.ipynb', 'load_in_model_just_confident_misclassifications.ipynb', '__init__.py', '.ipynb_checkpoints', 'load_in_model_just_clustering_DV_1.ipynb', '__pycache__', 'load_in_model_complete.ipynb']


In [8]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN

import torch

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import math

In [9]:
from utils.cache_directory_utils import CacheDirectoryManager
from data.load_data import mount_drive, load_data
from models.transformer_model import CaseTransformerModel
from models.transformer_creation import create_vocabularies_from_data
from config import get_model_config
from analysis.pca_vectors import (
    extract_all_pre_classifier_vectors,
    plot_multilabel_cluster_heatmap,
)
from notebook_helpers import prepare_sequences_and_labels

## Config

In [10]:
BASE_CACHE  = Path('/data/krosen5/cache/transformers/balanced_8')
MODELS_DIR  = BASE_CACHE / 'saved_models'
OUTPUT_DIR  = BASE_CACHE / 'pca_best_fold'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_FOLD_IDX = 0        # 0-indexed — change to your best fold
N_FOLDS       = 5
RANDOM_SEED   = 42
N_CLUSTERS    = 4        # was 6 — DV subset is smaller, so fewer clusters
bert_out_dim  = 768
cuda_index    = 0

# DV filter: case is kept if ANY charge name on ANY proceeding contains 'DOMESTIC'
DV_KEYWORD = 'DOMESTIC'

## Device, label processor, sequences, fold splits

In [11]:
computation_device = torch.device(
    f'cuda:{cuda_index}' if torch.cuda.is_available() else 'cpu'
)
print(f'Using device: {computation_device}')

model_cache_manager = CacheDirectoryManager(base_path=BASE_CACHE)
model_config = get_model_config()

with open(MODELS_DIR / 'label_processor.pkl', 'rb') as f:
    label_processor = pickle.load(f)
print(f'[ok] Loaded label processor ({len(label_processor.label_names)} labels)')

sequences_path = model_cache_manager.get_cache_dir('datasets') / 'sequences_and_labels.pkl'
with open(sequences_path, 'rb') as f:
    sequences, label_sequences_raw = pickle.load(f)
print(f'[ok] Loaded {len(sequences)} sequences')

filtered_training_data = list(zip(sequences, label_sequences_raw))
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
fold_val_sets = [
    [filtered_training_data[i] for i in val_idx]
    for _, val_idx in kf.split(filtered_training_data)
]
print(f'[ok] Recreated {N_FOLDS} fold splits')
print(f'   Best fold {BEST_FOLD_IDX + 1} val size: {len(fold_val_sets[BEST_FOLD_IDX])}')

Using device: cuda:0
[ok] Loaded label processor (20 labels)
[ok] Loaded 2533 sequences
[ok] Recreated 5 fold splits
   Best fold 1 val size: 507


## Filter the val set to DV cases

Done **before** extraction so PCA / t-SNE / KMeans only see DV cases. A case is kept if any proceeding has a charge whose name contains `DOMESTIC` (charge names in this dataset are uppercase).

In [12]:
def case_touches_dv(seq, keyword=DV_KEYWORD):
    """True if ANY charge on ANY proceeding in the case contains `keyword`."""
    for proc in seq:
        charges = proc.get('charges', {}) or {}
        for _, info in charges.items():
            name = (info.get('name') or '') if isinstance(info, dict) else str(info)
            if keyword in name:
                return True
    return False

best_fold_val = fold_val_sets[BEST_FOLD_IDX]
val_data_dv = [(seq, lbls) for (seq, lbls) in best_fold_val if case_touches_dv(seq)]
print(f'DV subset: {len(val_data_dv)} / {len(best_fold_val)} cases '
      f'({100*len(val_data_dv)/max(1,len(best_fold_val)):.1f}%)')

DV subset: 111 / 507 cases (21.9%)


### (Optional) Sanity check — which charge names matched?

In [13]:
hits = set()
for seq, _ in best_fold_val:
    for proc in seq:
        for _, info in (proc.get('charges') or {}).items():
            name = info.get('name', '') if isinstance(info, dict) else str(info)
            if DV_KEYWORD in name:
                hits.add(name)
print(f'{len(hits)} unique charge names matched:')
for n in sorted(hits):
    print(' ', n)

11 unique charge names matched:
  CRIMINAL DAMAGE/DOMESTIC
  DOMESTIC ABUSE AGG ASSAULT
  DOMESTIC ABUSE AGGRAVATED ASSAULT
  DOMESTIC ABUSE BATTERY
  DOMESTIC ABUSE BATTERY FIRST OFFENSE
  DOMESTIC ABUSE BATTERY INV STRANGULATION
  DOMESTIC ABUSE BATTERY-VICTIM PREGNANT
  DOMESTIC ABUSE CHILD ENDANGEMENT LAW
  DOMESTIC ABUSE SECOND OFFENSE
  DOMESTIC ABUSE THIRD OFFENSE
  DOMESTIC ABUSE WITH A DANGEROUS WEAPON


## Vocabularies + load best-fold model

In [14]:
drive_root = mount_drive('tulane-courts-shared/v2')
court_data = load_data(drive_root)
charge_vocab, mag_vocab = create_vocabularies_from_data(court_data)
del court_data  # free memory

✓ Found Google Drive mount at /data/krosen5/gdrive-all
✓ Google Drive mount is accessible
✓ Data directory found: /data/krosen5/gdrive-all/tulane-courts-shared/v2
Creating charge vocabulary from charges DataFrame with 2139 unique charges
Found 5 unique magistrate sections in mag data.
Creating magistrate vocabulary of size 5


In [15]:
bond_dim = model_config.NUM_BOND_CLASSES  # 9
sample_pt = MODELS_DIR / f'k_fold_fold_{BEST_FOLD_IDX + 1}' / \
            f'final_model_mlb_fold_{BEST_FOLD_IDX + 1}.pt'
sample_sd = torch.load(sample_pt, map_location='cpu')
fuse_in   = sample_sd['fuse.0.weight'].shape[1]  # 1548
FIXED_DIMS = 32 + 192 + 768 + 32 + 96 + 192 + 1 + 1  # 1314
correct_temporal_dim = fuse_in - FIXED_DIMS - (bond_dim * 2)  # 216
print(f'[ok] temporal_dim={correct_temporal_dim}')

[ok] temporal_dim=216


In [16]:
print(f'\n{"-"*60}')
print(f'  Loading fold {BEST_FOLD_IDX + 1} (best fold)')
print(f'{"-"*60}')

model = CaseTransformerModel(
    charge_vocab=charge_vocab,
    mag_vocab=mag_vocab,
    bert_cache_dir=model_cache_manager.get_cache_dir('bert'),
    bert_out_dim=bert_out_dim,
    num_labels=model_config.num_labels,
    bond_dim=bond_dim,
    temporal_dim=correct_temporal_dim,
    cache_manager=model_cache_manager,
)
model.load_state_dict(torch.load(sample_pt, map_location=computation_device))
model.eval()
model.to(computation_device)
print(f'[ok] Loaded {sample_pt.name}')


------------------------------------------------------------
  Loading fold 1 (best fold)
------------------------------------------------------------
[ok] Loaded final_model_mlb_fold_1.pt


## Extract pre-classifier vectors on the DV subset

In [17]:
_saved_min_year = model_config.min_year
_saved_max_year = model_config.max_year
model_config.update_date_range(2008, 2025)
print(f'  Config forced: temporal_dim={model_config.temporal_dim}')

try:
    vectors, final_labels, case_labels_matrix, metadata = \
        extract_all_pre_classifier_vectors(
            model,
            val_data_dv,                 # <-- DV-filtered subset
            label_processor,
            computation_device,
            use_teacher_forcing=True,
            verbose=True,
        )
finally:
    model_config.update_date_range(_saved_min_year, _saved_max_year)
    print(f'[ok] Config restored: temporal_dim={model_config.temporal_dim}')

del model
torch.cuda.empty_cache()

print(f'\n[ok] Extracted {vectors.shape[0]} embeddings  (dim={vectors.shape[1]})')

Date range updated: 2008 to 2025
  Config forced: temporal_dim=216


Extracting pre-classifier vectors for 111 samples: 100%|██████████| 111/111 [00:01<00:00, 91.42it/s] 

Extracted 111 vectors of dimension 896
Date range updated: 1990 to 2025
[ok] Config restored: temporal_dim=432

[ok] Extracted 111 embeddings  (dim=896)


In [18]:
vectors = StandardScaler().fit_transform(vectors)
vectors = normalize(vectors, norm='l2')
print('[ok] Normalised (StandardScaler + L2)')

case_labels_arr = np.asarray(case_labels_matrix)

[ok] Normalised (StandardScaler + L2)


## PCA -> t-SNE -> KMeans

In [19]:
print('\nRunning PCA (50 components)...')
n_pca = min(50, vectors.shape[0], vectors.shape[1])
pca = PCA(n_components=n_pca, random_state=RANDOM_SEED)
vecs_50d = pca.fit_transform(vectors)
explained = pca.explained_variance_ratio_.sum() * 100
print(f'[ok] PCA done -- {explained:.1f}% variance in {n_pca} PCs')

print('Running t-SNE...')
# Perplexity must be < n_samples; clamp it for the smaller DV subset.
perplexity = min(40, max(5, vectors.shape[0] // 4))
projected_2d = TSNE(
    n_components=2,
    perplexity=perplexity,
    learning_rate='auto',
    init='pca',
    random_state=RANDOM_SEED,
).fit_transform(vecs_50d)
print(f'[ok] t-SNE done -- shape: {projected_2d.shape}  (perplexity={perplexity})')


Running PCA (50 components)...
[ok] PCA done -- 97.3% variance in 50 PCs
Running t-SNE...
[ok] t-SNE done -- shape: (111, 2)  (perplexity=27)


In [20]:
km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
km_labels = km.fit_predict(projected_2d)
print(f'[ok] KMeans: {N_CLUSTERS} clusters')

[ok] KMeans: 4 clusters


## Plot helpers

In [21]:
def scatter_2d(xy, colours, legend_labels, title, output_path,
               xlabel='t-SNE dim 1', ylabel='t-SNE dim 2',
               s=18, alpha=0.6):
    fig, ax = plt.subplots(figsize=(12, 9))
    unique = sorted(set(colours))
    cmap = cm.get_cmap('tab10', len(unique))
    for i, val in enumerate(unique):
        mask = np.array(colours) == val
        label = legend_labels[i] if legend_labels else str(val)
        ax.scatter(xy[mask, 0], xy[mask, 1],
                   c=[cmap(i)], label=label,
                   s=s, alpha=alpha, edgecolors='w', linewidths=0.2)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    print(f'[saved] Saved: {output_path}')
    plt.close(fig)


def plot_label_grid_colored_by_cluster(
    projected_2d, label_matrix, label_names, cluster_labels, output_path,
    title='t-SNE Embeddings -- Label Presence', max_cols=5,
):
    label_matrix   = np.asarray(label_matrix)
    cluster_labels = np.asarray(cluster_labels)
    n_samples, n_labels = label_matrix.shape
    unique_clusters = sorted(np.unique(cluster_labels))
    n_real = sum(1 for c in unique_clusters if c != -1)
    cmap = cm.get_cmap('tab20' if n_real <= 20 else 'nipy_spectral', max(n_real, 1))
    cl_to_idx = {cl: i for i, cl in enumerate(unique_clusters)}

    n_cols = min(max_cols, n_labels)
    n_rows = math.ceil(n_labels / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for i, name in enumerate(label_names):
        ax = axes[i]
        mask_pos = label_matrix[:, i] == 1
        n_pos = int(mask_pos.sum())
        pct = 100 * n_pos / n_samples if n_samples else 0.0
        ax.scatter(projected_2d[~mask_pos, 0], projected_2d[~mask_pos, 1],
                   c='lightgrey', s=8, alpha=0.25, edgecolors='none')
        for cl in unique_clusters:
            mask = mask_pos & (cluster_labels == cl)
            if not mask.any():
                continue
            fc = 'dimgrey' if cl == -1 else [cmap(cl_to_idx[cl])]
            ax.scatter(projected_2d[mask, 0], projected_2d[mask, 1],
                       c=fc, s=14, alpha=0.35 if cl == -1 else 0.75,
                       edgecolors='white', linewidths=0.2)
        ax.set_title(f'{name[:30]}\n(n={n_pos:,}, {pct:.1f}%)', fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])

    for j in range(n_labels, len(axes)):
        axes[j].set_visible(False)

    handles = [
        plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor='dimgrey' if cl == -1 else cmap(cl_to_idx[cl]),
                   markeredgecolor='white', markersize=7,
                   label='Noise' if cl == -1 else f'Cluster {cl}')
        for cl in unique_clusters
    ]
    fig.legend(handles=handles, loc='upper center',
               bbox_to_anchor=(0.5, 1.01),
               ncol=min(len(unique_clusters), 6), fontsize=9)
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.05)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    print(f'[saved] Saved: {output_path}')
    plt.close(fig)

## Save the three DV plots

In [22]:
fold_label = f'Fold {BEST_FOLD_IDX + 1} (best), DV charges only (n={vectors.shape[0]})'

scatter_2d(
    projected_2d,
    km_labels.tolist(),
    [f'Cluster {i}' for i in range(N_CLUSTERS)],
    title=f't-SNE Embeddings -- KMeans Clusters ({fold_label})',
    output_path=OUTPUT_DIR / 'tsne_kmeans_clusters_DV.png',
)

plot_label_grid_colored_by_cluster(
    projected_2d=projected_2d,
    label_matrix=case_labels_arr,
    label_names=label_processor.label_names,
    cluster_labels=km_labels,
    output_path=OUTPUT_DIR / 'tsne_label_grid_DV.png',
    title=f't-SNE Embeddings ({fold_label}) -- Label Presence by Cluster',
)

plot_multilabel_cluster_heatmap(
    km_labels, case_labels_arr, label_processor.label_names,
    output_path=OUTPUT_DIR / 'tsne_multilabel_heatmap_DV.png',
)

print(f'\n[ok] Done. DV plots saved to {OUTPUT_DIR}')
print(f'   Best fold:  {BEST_FOLD_IDX + 1}')
print(f'   DV samples: {projected_2d.shape[0]}')
print(f'   Clusters:   {N_CLUSTERS}')
print(f'   PCA var:    {explained:.1f}%')

[saved] Saved: /data/krosen5/cache/transformers/balanced_8/pca_best_fold/tsne_kmeans_clusters_DV.png
[saved] Saved: /data/krosen5/cache/transformers/balanced_8/pca_best_fold/tsne_label_grid_DV.png
Saved multi-label heatmap to /data/krosen5/cache/transformers/balanced_8/pca_best_fold/tsne_multilabel_heatmap_DV.png

MULTI-LABEL CLUSTER SUMMARY
----------------------------------------------------------------------
  Cluster 0 (17 samples): FIRST_APPEARANCE_HEARING_OCCURRED (100%, n=17), CHARGES_REFUSED (94%, n=16), BOND_SET (94%, n=16), DOMESTIC_STAY_AWAY_ORDER_IMPOSED (88%, n=15), THE_COURT_APPOINTED_THE_ORLEANS_PUBLIC_… (71%, n=12), PROBABLE_CAUSE_FOUND (24%, n=4), THE_STATE_WAS_NOT_PRESENT (18%, n=3), CONTINUANCE_OCCURRED (18%, n=3), PLED_GUILTY_TO_CHARGES (18%, n=3), ANKLE_MONITORING_IMPOSED (18%, n=3), BOND_NOT_SET (18%, n=3), SENTENCE_THAT_DOES_NOT_INCLUDE_JAIL_TIM… (12%, n=2), PRE_TRIAL_CONFERENCE_HELD (6%, n=1), PROBABLE_CAUSE_NOT_FOUND (6%, n=1), SENTENCE_THAT_INCLUDES_JAIL_TIME_